# Experiments for GitHub Issues Classification
## Author: Achintya Kattemalavadi (akattema@berkeley.edu)
This notebook contains all the code to train, test, and evaluate various models covered in the GitHub issues classification project.

## Installing packages

In [ ]:
!pip install -q pandas
!pip install -q matplotlib
!pip install -q seaborn
!pip install -q scikit-learn
!pip install -q transformers
!pip install -q torch
!pip install -q torchvision
!pip install -q torchaudio
!pip install -q datasets
!pip install -q ipykernel
!pip install -q accelerate
!pip install -q evaluate
!pip install -q rouge_score

## Imports

In [ ]:
# Imports
import json
import os
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report
from transformers import (
    Trainer,
    TrainingArguments,
    get_scheduler,
    RobertaTokenizer,
    BertTokenizerFast,
    BertForSequenceClassification,
    DataCollatorForSeq2Seq,
    T5ForConditionalGeneration,
    T5TokenizerFast,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from transformers.optimization import Adafactor
import torch
from torch.optim import AdamW
from torch.utils.data.dataloader import default_collate
import numpy as np
from datasets import Dataset
import evaluate
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

## Setting base directory
This directory is where the datasets are located, and where training/testing results will be stored.

In [ ]:
BASE_DIR = "."

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = f"drive/MyDrive/College/Grad School/UC Berkeley MIDS/Coursework/Spring 2025/DATASCI 266/final project/DATASCI-266-Final-Project"

## Use GPU (Mac-specific)

In [ ]:
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# print(f"Using device: {device}")

## Load datasets
Load in data from the JSON files and set them up as HuggingFace datasets.

In [ ]:
# Load the data
train_issues_list = []
train_issues_list_file = f"{BASE_DIR}/data/dataset_rebalanced_reformatted_train.json"
with open(train_issues_list_file, "r", encoding="utf-8") as f:
    train_issues_list = json.load(f)

val_issues_list = []
val_issues_list_file = f"{BASE_DIR}/data/dataset_rebalanced_reformatted_val.json"
with open(val_issues_list_file, "r", encoding="utf-8") as f:
    val_issues_list = json.load(f)

test_issues_list = []
test_issues_list_file = f"{BASE_DIR}/data/dataset_rebalanced_reformatted_test.json"
with open(test_issues_list_file, "r", encoding="utf-8") as f:
    test_issues_list = json.load(f)

In [ ]:
# Get the number of labels
num_labels = len(train_issues_list[0]["labels"])

In [ ]:
# Create the datasets
train_iss_ds = Dataset.from_list(train_issues_list)
val_iss_ds = Dataset.from_list(val_issues_list)
test_iss_ds = Dataset.from_list(test_issues_list)

# BERT Baseline

## Tokenize the dataset

In [ ]:
# Load tokenizer and tokenize the data
bert_fast_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return bert_fast_tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

train_iss_ds_tokenized = train_iss_ds.map(tokenize_function, batched=True)
val_iss_ds_tokenized = val_iss_ds.map(tokenize_function, batched=True)
test_iss_ds_tokenized = test_iss_ds.map(tokenize_function, batched=True)

In [ ]:
train_iss_ds_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_iss_ds_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_iss_ds_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

## Method to compute relevant metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Apply sigmoid to convert logits to probabilities, then threshold at 0.5
    preds = (torch.sigmoid(torch.tensor(logits)) > 0.5).numpy()
    labels = np.array(labels)

    # Compute micro and macro precision, recall, and F1 using scikit-learn.
    micro_precision = precision_score(labels, preds, average="micro")
    micro_recall = recall_score(labels, preds, average="micro")
    micro_f1 = f1_score(labels, preds, average="micro")

    macro_precision = precision_score(labels, preds, average="macro")
    macro_recall = recall_score(labels, preds, average="macro")
    macro_f1 = f1_score(labels, preds, average="macro")

    accuracy = (preds == labels).mean()
    subset_accuracy = accuracy_score(labels, preds)

    return {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "subset_accuracy": subset_accuracy
    }

## See pre-trained BERT model details
Load in the `bert-base-uncased` model and view its components.

In [ ]:
# Load the model (configure for multi-label classification)
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)
print(model)

## Custom trainer class
This trainer allows changing optimizer types in a hyperparmeter grid search when training.

NOTE: In this notebook, only the `AdamW` optimizer is being used, but future exploration could involve trying different optimizers such as `Adafactor`.

In [ ]:
# Custom Trainer to support multiple optimizer types
class MyTrainer(Trainer):
    def __init__(self, *args, optimizer_type="adamw", **kwargs):
        self.optimizer_type = optimizer_type
        super().__init__(*args, **kwargs)

    def create_optimizer_and_scheduler(self, num_training_steps: int):
        # If optimizer or scheduler is not set, create them
        if self.optimizer is None or self.lr_scheduler is None:
            if self.optimizer_type.lower() == "adafactor":
                optimizer = Adafactor(
                    self.model.parameters(),
                    scale_parameter=True,
                    relative_step=True,
                    lr=self.args.learning_rate,
                    weight_decay=self.args.weight_decay,
                    eps=(1e-30, 1e-3)
                )
            else:
                optimizer = AdamW(
                    self.model.parameters(),
                    lr=self.args.learning_rate,
                    weight_decay=self.args.weight_decay
                )

            scheduler = get_scheduler(
                name=self.args.lr_scheduler_type,
                optimizer=optimizer,
                num_warmup_steps=self.args.warmup_steps if hasattr(self.args, "warmup_steps") else 0,
                num_training_steps=num_training_steps,
            )
            # Assign the created optimizer and scheduler to the Trainer instance
            self.optimizer = optimizer
            self.lr_scheduler = scheduler
        return self.optimizer, self.lr_scheduler

## BERT baseline hyperparameters grid

In [ ]:
# Define hyperparameter grid for grid search
baseline_hyperparams_grid = {
    "per_device_train_batch_size": [8, 16],
    "num_train_epochs": [3, 4],
    "freeze_layers": [0, 6],  # 0: no freezing, 6: freeze first 6 encoder layers
}

# Create all combinations from the grid
bl_hp_keys, bl_hp_values = zip(*baseline_hyperparams_grid.items())
bl_hyperparams_combinations = [dict(zip(bl_hp_keys, v)) for v in itertools.product(*bl_hp_values)]

## Train the baseline BERT model
Loop through each combination of hyperparameters and train a model.

In [ ]:
os.makedirs(f"{BASE_DIR}/model_results/baseline_best_models", exist_ok=True)

# Iterate over each hyperparameter combination
for idx, params in enumerate(bl_hyperparams_combinations):

    print(f"Training configuration {idx+1}/{len(bl_hyperparams_combinations)}: {params}")

    # Create a unique output directory for this configuration
    output_dir = f"{BASE_DIR}/model_results/baseline_results/config_{idx}"
    os.makedirs(output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=params["per_device_train_batch_size"],
        num_train_epochs=params["num_train_epochs"],
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="micro_f1",
        logging_steps=10,
        logging_dir=f"{output_dir}/logs"
    )

    # Reinitialize model for each configuration
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=num_labels,
        problem_type="multi_label_classification"
    )

    # Mac specific:
    # model.to(device)

    # Apply layer freezing if specified (e.g., freeze first 6 layers)
    freeze_layers = params["freeze_layers"]
    if freeze_layers > 0:
        # Bert encoder layers are in model.bert.encoder.layer (index 0 to 11 for bert-base-uncased)
        for param in model.bert.encoder.layer[:freeze_layers].parameters():
            param.requires_grad = False

    # Initialize custom trainer
    trainer = MyTrainer(
        model=model,
        args=training_args,
        train_dataset=train_iss_ds_tokenized,
        eval_dataset=val_iss_ds_tokenized,
        compute_metrics=compute_metrics
    )

    # Train the model for the current configuration
    trainer.train()

    # Save the best model from this configuration
    best_model_dir = f"{BASE_DIR}/model_results/baseline_best_models/config_{idx}"
    os.makedirs(best_model_dir, exist_ok=True)
    trainer.save_model(best_model_dir)
    print(f"Saved best model for configuration {idx+1} to {best_model_dir}\n")

Loss is average binary cross entropy loss across all labels

## List of the label names
In order of the one-hot representation.

In [ ]:
lab_names = [
    "accessibility",
    "release",
    "dependency",
    "good_first_issue",
    "webview",
    "oslinux",
    "revert",
    "bug",
    "documentation",
    "featurerequest",
    "feature",
    "help_wanted",
    "test",
    "ui",
    "api",
    "enhancement",
    "regression",
    "security"
]

## BERT model evaluation
Using the test dataset.

In [ ]:
# Evaluate models
success_models = []
failed_models = []
for i in range(len(bl_hyperparams_combinations)):

    try:
        # Load model
        curr_model_dir = f"{BASE_DIR}/model_results/baseline_best_models/config_{i}"
        curr_model = BertForSequenceClassification.from_pretrained(curr_model_dir)

        # Set up minimal training args
        test_out_dir = f"{BASE_DIR}/model_results/baseline_test_results/config_{i}"
        train_args = TrainingArguments(
            output_dir=test_out_dir,
            per_device_eval_batch_size=8
        )

        trainer = Trainer(
            model=curr_model,
            args=train_args,
            eval_dataset=test_iss_ds_tokenized,
            compute_metrics=compute_metrics
        )

        # Run evaluation on the test set
        results = trainer.evaluate()
        with open(f"{BASE_DIR}/model_results/baseline_test_results/config_{i}/test_results.json", "w+") as res_f:
            json.dump(results, res_f, indent=4)

        print(f"Model {i} test set evaluation results:", results)

        # Figure out performance on each label
        test_preds = trainer.predict(test_iss_ds_tokenized)
        logits = test_preds.predictions
        labs = test_preds.label_ids

        bin_preds = (torch.sigmoid(torch.tensor(logits)) > 0.5).numpy()

        report = classification_report(
            labs,
            bin_preds,
            target_names=lab_names,
            output_dict=True
        )
        with open(f"{BASE_DIR}/model_results/baseline_test_results/config_{i}/test_classification_report.json", "w+") as cls_f:
            json.dump(report, cls_f, indent=4)

        print(report)

        success_models.append(i)

    except Exception as e:
        print(f"Issue evaluating model {i}, skipping: {e}")
        failed_models.append(i)

print(f"Suceeded evaluating the following baseline models: {success_models}")
print(f"Failed evaluating the following baseline models: {failed_models}")

# RoBERTa Model

## Tokenize the dataset

In [ ]:
# Load tokenizer and tokenize the data
roberta_tokenizer = RobertaTokenizer.from_pretrained("FacebookAI/roberta-base")

def roberta_tokenize_function(example):
    return roberta_tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

train_iss_ds_rob_tokenized = train_iss_ds.map(roberta_tokenize_function, batched=True)
val_iss_ds_rob_tokenized = val_iss_ds.map(roberta_tokenize_function, batched=True)
test_iss_ds_rob_tokenized = test_iss_ds.map(roberta_tokenize_function, batched=True)

## See pre-trained RoBERTa model details
Load in the `roberta-base` model and view its components.

In [ ]:
# Load RoBERTa model (configure for multi-label classification) and print layers
model = BertForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)
print(model)

## RoBERTa hyperparameters grid

In [ ]:
# Define hyperparameter grid for grid search
roberta_hyperparams_grid = {
    "per_device_train_batch_size": [8, 16],
    "num_train_epochs": [3, 4],
    "freeze_layers": [0, 6],  # 0: no freezing, 6: freeze first 6 encoder layers
}

# Create all combinations from the grid
rb_hp_keys, rb_hp_values = zip(*roberta_hyperparams_grid.items())
rb_hyperparams_combinations = [dict(zip(rb_hp_keys, v)) for v in itertools.product(*rb_hp_values)]

## Train the RoBERTa model

In [ ]:
# Roberta training
os.makedirs(f"{BASE_DIR}/roberta_best_models", exist_ok=True)

# Iterate over each hyperparameter combination
for idx, params in enumerate(rb_hyperparams_combinations):


    print(f"Training configuration {idx+1}/{len(rb_hyperparams_combinations)}: {params}")

    # Create a unique output directory for this configuration
    output_dir = f"{BASE_DIR}/roberta_results/config_{idx}"
    os.makedirs(output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=params["per_device_train_batch_size"],
        num_train_epochs=params["num_train_epochs"],
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="micro_f1",
        logging_steps=10,
        logging_dir=f"{output_dir}/logs",
        prediction_loss_only=True
    )

    # Reinitialize model for each configuration
    model = BertForSequenceClassification.from_pretrained(
        "roberta-base",
        num_labels=num_labels,
        problem_type="multi_label_classification"
    )

    # Mac-specific
    # model.to(device)

    freeze_layers = params["freeze_layers"]
    if freeze_layers > 0:
        for param in model.bert.encoder.layer[:freeze_layers].parameters():
            param.requires_grad = False

    # Initialize our custom trainer with the chosen optimizer type
    trainer = MyTrainer(
        model=model,
        args=training_args,
        train_dataset=train_iss_ds_rob_tokenized,
        eval_dataset=val_iss_ds_rob_tokenized,
        compute_metrics=compute_metrics
    )

    # Train the model for the current configuration
    trainer.train()

    # Save the best model from this configuration
    best_model_dir = f"{BASE_DIR}/roberta_best_models/config_{idx}"
    os.makedirs(best_model_dir, exist_ok=True)
    trainer.save_model(best_model_dir)
    print(f"Saved best model for configuration {idx+1} to {best_model_dir}\n")

## Evaluate the RoBERTa models

In [ ]:
# Evaluate models
success_roberta_models = []
failed_roberta_models = []
for i in range(len(rb_hyperparams_combinations)):

    try:
        # Load model
        curr_model_dir = f"{BASE_DIR}/roberta_best_models/config_{i}"
        curr_model = BertForSequenceClassification.from_pretrained(curr_model_dir)

        # Set up minimal training args
        test_out_dir = f"{BASE_DIR}/roberta_test_results/config_{i}"
        train_args = TrainingArguments(
            output_dir=test_out_dir,
            per_device_eval_batch_size=8
        )

        trainer = Trainer(
            model=curr_model,
            args=train_args,
            eval_dataset=test_iss_ds_rob_tokenized,
            compute_metrics=compute_metrics
        )

        # Run evaluation on the test set
        results = trainer.evaluate()
        with open(f"{BASE_DIR}/roberta_test_results/config_{i}/test_results.json", "w+") as res_f:
            json.dump(results, res_f, indent=4)

        print(f"Model {i} test set evaluation results:", results)

        # Figure out performance on each label
        test_preds = trainer.predict(test_iss_ds_rob_tokenized)
        logits = test_preds.predictions
        labs = test_preds.label_ids

        bin_preds = (torch.sigmoid(torch.tensor(logits)) > 0.5).numpy()

        report = classification_report(
            labs,
            bin_preds,
            target_names=lab_names,
            output_dict=True
        )
        with open(f"{BASE_DIR}/roberta_test_results/config_{i}/test_classification_report.json", "w+") as cls_f:
            json.dump(report, cls_f, indent=4)

        print(report)

        success_models.append(i)

    except Exception as e:
        print(f"Issue evaluating model {i}, skipping: {e}")
        failed_models.append(i)

print(f"Suceeded evaluating the following roberta models: {success_roberta_models}")
print(f"Failed evaluating the following roberta models: {failed_roberta_models}")

# T5 Model
## Method to convert the dataset to a text-to-text format

In [ ]:
# Converter for data list to correct format

def t5_ds_converter(data_list, out_filename=None):
  """
  Takes in data in the following format:

  [
    {
      "text": "Title: ... [SEP] Body: ... [SEP] Comments: [...]",
      "labels": [
        0.0,
        1.0,
        0.0
        ...
      ]
    }
  ]

  and converts it to the following format:

  [
    {
      "input_text": "Title: ... [SEP] Body: ... [SEP] Comments: [...]",
      "target_text": "label_1 label_3 ..."
    }
  ]

  TODO: should we copy? Or keep in place?
  """

  final_data = []

  for dat in data_list:

    new_dat = {}

    # Change the name of the text var
    new_dat["input_text"] = dat["text"]

    # Target labels list
    target_labs = []

    # Get the label booleans and based on each append the label if present
    labs = dat["labels"]
    for i in range(len(labs)):
      if labs[i] > 0.5:
        target_labs.append(lab_names[i])

    # Final target text is the joined list as a comma-separated string
    new_dat["target_text"] = " ".join(target_labs)
    final_data.append(new_dat)

  # Write dataset
  if out_filename:
    with open(out_filename, "w+") as d_f:
      json.dump(final_data, d_f, indent=4)

  # Return a HuggingFace dataset
  return Dataset.from_list(final_data)

train_iss_t5_ds = t5_ds_converter(train_issues_list, f"{BASE_DIR}/data/dataset_reformatted_t5_train.json")
val_iss_t5_ds = t5_ds_converter(val_issues_list, f"{BASE_DIR}/data/dataset_reformatted_t5_val.json")
test_iss_t5_ds = t5_ds_converter(test_issues_list, f"{BASE_DIR}/data/dataset_reformatted_t5_test.json")

## Tokenize the dataset for T5

In [ ]:
# Load the T5 tokenizer
t5_tokenizer = T5TokenizerFast.from_pretrained("t5-base")

# Tokenize for T5
def tokenize_t5(example):

    # Tokenize input text
    model_inputs = t5_tokenizer(example["input_text"], truncation=True, padding="max_length", max_length=512)

    # Tokenize target text
    with t5_tokenizer.as_target_tokenizer():
        labels = t5_tokenizer(example["target_text"], truncation=True, padding="max_length", max_length=128)

    # Replace all pad tokens in labels by -100 to ignore them in the loss
    labels["input_ids"] = [token if token != t5_tokenizer.pad_token_id else -100 for token in labels["input_ids"]]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply tokenization over the datasets in batches
train_iss_t5_ds_tokenized = train_iss_t5_ds.map(tokenize_t5, batched=True)
val_iss_t5_ds_tokenized = val_iss_t5_ds.map(tokenize_t5, batched=True)
test_iss_t5_ds_tokenized = test_iss_t5_ds.map(tokenize_t5, batched=True)

train_iss_t5_ds_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_iss_t5_ds_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_iss_t5_ds_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

## See pre-trained T5 model details

In [ ]:
# Load T5 model and print layers
model = T5ForConditionalGeneration.from_pretrained(
    "t5-base"
)
print(model)

## T5 hyperparameters grid

In [ ]:
# T5 Hyperparameters grid
t5_hyperparams_grid = {
    "per_device_train_batch_size": [8, 16],
    "beams": [1, 6],
    "num_train_epochs": [3, 4],
    "freeze_layers": [6],  # 0: no freezing, 6: freeze first 6 encoder and decoder layers
}

# Create all combinations from the grid
t5_hp_keys, t5_hp_values = zip(*t5_hyperparams_grid.items())
t5_hyperparams_combinations = [dict(zip(t5_hp_keys, v)) for v in itertools.product(*t5_hp_values)]

## Custom metrics function for T5
In addition to ROUGE and BLEU metrics, will also calculate classification metrics through the following steps:


1. Split each prediction and label string by whitespace
2. Create a one-hot list of numbers (0 or 1) representing whether each label is present in the prediction and the actual
3. Calculate the same metrics as for the BERT models



In [ ]:
# Load the ROUGE metric from the evaluate library.
rouge = evaluate.load("rouge")

In [ ]:
# T5 compute metrics
def compute_t5_metrics(eval_pred):
    """
    Compute evaluation metrics for a generative multi-label model.
    Assumes predictions and references are strings, each representing a comma-separated
    list of labels. The function computes:
      - Micro and Macro Precision, Recall, and F1.
      - Mean (element-wise) accuracy.
      - Subset accuracy (exact match per sample).
      - ROUGE-1, ROUGE-2, and ROUGE-L scores.
      - Average BLEU score.
    """
    predictions, labels = eval_pred

    # Decode predictions and references if needed
    # If the first element is not a string assume they are token IDs
    if not isinstance(predictions[0], str):
        predictions = t5_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    if not isinstance(labels[0], str):
        labels = t5_tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    rouge = evaluate.load("rouge")
    rouge_result = rouge.compute(predictions=predictions, references=labels)

    # Compute BLEU scores per sample
    bleu_scores = []
    smoothie = SmoothingFunction().method4
    for pred, ref in zip(predictions, labels):
        try:
            bleu = sentence_bleu([ref], pred, smoothing_function=smoothie)
        except Exception:
            bleu = 0.0
        bleu_scores.append(bleu)
    avg_bleu = np.mean(bleu_scores)

    # Convert each prediction and reference string into a list of labels
    def split_labels(s):
        return [token.strip().lower() for token in s.split() if token.strip()]

    pred_lists = [split_labels(pred) for pred in predictions]
    ref_lists  = [split_labels(ref) for ref in labels]

    # Convert each list into a binary vector based on lab_names
    pred_binary = []
    label_binary = []

    for pred, ref in zip(pred_lists, ref_lists):
        pred_vec = [1 if name in pred else 0 for name in lab_names]
        label_vec = [1 if name in ref else 0 for name in lab_names]
        pred_binary.append(pred_vec)
        label_binary.append(label_vec)

    pred_binary = np.array(pred_binary)
    label_binary = np.array(label_binary)

    # Compute micro and macro precision, recall, and F1
    micro_precision = precision_score(label_binary, pred_binary, average="micro", zero_division=0)
    micro_recall = recall_score(label_binary, pred_binary, average="micro", zero_division=0)
    micro_f1 = f1_score(label_binary, pred_binary, average="micro", zero_division=0)

    macro_precision = precision_score(label_binary, pred_binary, average="macro", zero_division=0)
    macro_recall = recall_score(label_binary, pred_binary, average="macro", zero_division=0)
    macro_f1 = f1_score(label_binary, pred_binary, average="macro", zero_division=0)

    # Mean accuracy
    accuracy = (pred_binary == label_binary).mean()

    # Subset accuracy
    subset_accuracy = accuracy_score(label_binary, pred_binary)

    # Return metrics in dict
    metrics = {
        "rouge1": rouge_result.get("rouge1", None),
        "rouge2": rouge_result.get("rouge2", None),
        "rougeL": rouge_result.get("rougeL", None),
        "bleu": avg_bleu,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "subset_accuracy": subset_accuracy
    }
    print(metrics)
    return metrics

In [ ]:
# T5 training

# Directory for saving best T5 models for each configuration
os.makedirs("./t5_best_models", exist_ok=True)

# Data collator
data_collator = DataCollatorForSeq2Seq(t5_tokenizer, model=None)

# Loop over each hyperparameter configuration
for idx, params in enumerate(t5_hyperparams_combinations):
    print(f"Training T5 config {idx+1}/{len(t5_hyperparams_combinations)}: {params}")

    # Create a unique output directory for this configuration
    output_dir = f"{BASE_DIR}/t5_results/config_{idx}"
    os.makedirs(output_dir, exist_ok=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=params["per_device_train_batch_size"],
        num_train_epochs=params["num_train_epochs"],
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="micro_f1",
        logging_steps=10,
        logging_dir=f"{output_dir}/logs",
        predict_with_generate=True,
        generation_max_length=128,
        generation_num_beams=params["beams"]
    )

    # Reinitialize T5 model for each configuration
    model = T5ForConditionalGeneration.from_pretrained("t5-base")
    # model.to(device)

    # Freeze layers
    freeze_layers = params["freeze_layers"]
    if freeze_layers > 0:
      for layer_idx, layer in enumerate(model.encoder.block):
        if ilayer_dx < freeze_layers:
            for param in layer.parameters():
                param.requires_grad = False

      for layer_idx, layer in enumerate(model.decoder.block):
        if layer_idx < freeze_layers:
            for param in layer.parameters():
                param.requires_grad = False

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_iss_t5_ds_tokenized,
        eval_dataset=val_iss_t5_ds_tokenized,
        data_collator=data_collator,
        compute_metrics=compute_t5_metrics,
    )

    trainer.train()

    best_model_dir = f"{BASE_DIR}/t5_best_models/config_{idx}"
    os.makedirs(best_model_dir, exist_ok=True)
    trainer.save_model(best_model_dir)
    print(f"Saved best T5 model for config {idx+1} to {best_model_dir}\n")

In [ ]:
# Evaluate models
success_t5_models = []
failed_t5_models = []
for i, params in enumerate(t5_hyperparams_combinations):

    try:
        # Load model and tokenizer
        curr_model_dir = f"{BASE_DIR}/t5_best_models/config_{i}"
        curr_model = T5ForConditionalGeneration.from_pretrained(curr_model_dir)

        # Set up minimal training args
        test_out_dir = f"{BASE_DIR}/t5_test_results/config_{i}"
        train_args = Seq2SeqTrainingArguments(
            output_dir=test_out_dir,
            per_device_train_batch_size=params["per_device_train_batch_size"],
            logging_steps=10,
            logging_dir=f"{test_out_dir}/logs",
            predict_with_generate=True,
            generation_max_length=128,
            generation_num_beams=params["beams"],
            do_train=False,
            do_eval=False
        )

        trainer = Seq2SeqTrainer(
            model=curr_model,
            args=train_args,
            eval_dataset=test_iss_t5_ds_tokenized,
            compute_metrics=compute_t5_metrics
        )

        # Run evaluation on the test set
        results = trainer.evaluate()
        with open(f"{BASE_DIR}/t5_test_results/config_{i}/test_results.json", "w+") as res_f:
            json.dump(results, res_f, indent=4)

        print(f"Model {i} test set evaluation results:", results)

        # Figure out performance on each label
        test_preds = trainer.predict(test_iss_t5_ds_tokenized)
        decoded_preds = t5_tokenizer.batch_decode(test_preds.predictions, skip_special_tokens=True)

        bin_preds = []
        for pred in decoded_preds:
            pred_vec = [1.0 if name in pred else 0.0 for name in lab_names]
            bin_preds.append(pred_vec)

        bin_preds = np.array(bin_preds)
        true_labels = np.array([item["labels"] for item in test_issues_list])

        report = classification_report(
            true_labels,
            bin_preds,
            target_names=lab_names,
            output_dict=True
        )
        with open(f"{BASE_DIR}/t5_test_results/config_{i}/test_classification_report.json", "w+") as cls_f:
            json.dump(report, cls_f, indent=4)

        print(report)

        success_t5_models.append(i)

    except Exception as e:
        print(f"Issue evaluating model {i}, skipping: {e}")
        failed_t5_models.append(i)

print(f"Suceeded evaluating the following t5 models: {success_t5_models}")
print(f"Failed evaluating the following t5 models: {failed_t5_models}")